# Análises complementares da quarta sprint

Este caderno reúne as análises feitas ao fim da sprint que sustentam decisões registradas na documentação e no relatório da sprint, mas que não cabiam nos cadernos de modelagem e de comparação. São cinco.

1. A grade ampla por área, com vinte candidatos e escolha medida pelo aninhamento estrito, que sustenta a troca do estimador em Main Vent Sys e a recusa da mesma troca em Coil Temps.
2. O bootstrap sobre a formulação entregue, que dá o intervalo das métricas e uma segunda estimativa do erro.
3. O AutoML com FLAML, com a escolha de algoritmo e hiperparâmetros refeita em cada dobra.
4. O AutoML com PyCaret, que exige Python 3.11 e por isso roda em ambiente separado.
5. O efeito, no Oven HMI, de contar duas vezes o mesmo sensor, que explica por que resultados obtidos na base antiga pareciam melhores.

Todas seguem o protocolo do projeto: validação que retira uma configuração experimental inteira por vez, em dezessete dobras, com o patamar e a escala de cada componente calculados apenas no treino de cada dobra e o peso próximo ao limite. O caderno lê apenas arquivos CSV desta pasta e não consulta o conjunto de teste reservado.

## Como executar

No Colab, envie os sete arquivos dataset_<area>_13cat.csv, o limites_ativos.csv e o particao_treino_teste.csv, todos gerados pelo caderno tratamento_datasets_13cat desta pasta. A execução completa leva cerca de vinte e cinco minutos numa máquina de catorze núcleos; no Colab gratuito, que tem dois, o tempo é bem maior, e a seção 3 pode ter o orçamento reduzido na variável ORCAMENTO_FLAML.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
from joblib import Parallel, delayed
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor

ARQUIVOS = [f"dataset_{a}_13cat.csv" for a in ["oven_hmi", "ct_knobs", "main_vent_sys", "ipc_cooling",
                                               "mainback", "ipc_comp", "coil_temps"]]
ARQUIVOS += ["limites_ativos.csv", "particao_treino_teste.csv"]
try:
    from google.colab import files
    faltando = [f for f in ARQUIVOS if not os.path.exists(f)]
    if faltando:
        print("Envie os arquivos:", faltando)
        files.upload()
except ImportError:
    pass
assert all(os.path.exists(f) for f in ARQUIVOS), "execute a partir da pasta colabs_S4"

AREAS = ["oven_hmi", "ct_knobs", "main_vent_sys", "ipc_cooling", "mainback", "ipc_comp", "coil_temps"]
SEMENTES = (42, 1, 2)
MARGEM = 8.0
N_BLOCOS = 4
part = pd.read_csv("particao_treino_teste.csv")
DOBRA = dict(zip(part["run_idx"], part["dobra"]))
TESTE = set(part.loc[part["conjunto"] == "teste", "run_idx"])
LIM = pd.read_csv("limites_ativos.csv").set_index("tc_name")["limite_num"]
BASE_PREF = ("blower_direction_", "housing_", "board_position_", "run_case_", "deflector_hole_",
             "material_coil_support_")
# Formulacao entregue: estimador, codificacao das perdas e motivo do ensaio por area.
ENTREGA = {"oven_hmi": ("rf", "agrupado", "com"), "ct_knobs": ("rf", "detalhado", "com"),
           "main_vent_sys": ("et+gb", "detalhado", "com"), "ipc_cooling": ("rf", "agrupado", "com"),
           "mainback": ("rf", "detalhado", "com"), "ipc_comp": ("rf", "agrupado", "com"),
           "coil_temps": ("rf+gb", "detalhado", "com")}


def familia(c):
    t = c[len("losses_"):].lower()
    return next((n for k, n in (("no losses", "none"), ("simone", "simone"), ("miro", "miro")) if k in t), "other")


def entradas(area, trat, motivo):
    """Formato longo da area, sem o conjunto reservado. As colunas de ordem de acionamento ficam fora."""
    d = pd.read_csv(f"dataset_{area}_13cat.csv")
    perdas = [c for c in d.columns if c.startswith("losses_")]
    mot = [c for c in d.columns if c.startswith("reason_")]
    base = [c for c in d.columns if c.startswith(BASE_PREF)]
    if trat == "detalhado":
        P = d[perdas].to_numpy(float)
    else:
        fam = np.array([familia(c) for c in perdas]); A = d[perdas].to_numpy(float)
        P = np.column_stack([(A[:, fam == n].sum(axis=1) > 0) for n in sorted(set(fam.tolist()))]
                            + [A.sum(axis=1) == 0]).astype(float)
    X = np.hstack([d[base].to_numpy(float), P] + ([d[mot].to_numpy(float)] if motivo == "com" else []))
    alvos = [c for c in d.columns if not c.startswith(BASE_PREF + ("losses_", "reason_", "loss_profile_order_"))
             and d[c].notna().sum() >= 5]
    assert all(a in LIM.index for a in alvos), f"{area}: coluna de configuracao tratada como alvo"
    Xl, yl, rl, cl = [], [], [], []
    for j, a in enumerate(alvos):
        ok = d[a].notna().to_numpy()
        Xl.append(X[ok]); yl.append(d[a].to_numpy(float)[ok]); rl.append(np.arange(len(d))[ok]); cl.append(np.full(ok.sum(), j))
    Xl, yl, rl, cl = np.vstack(Xl), np.concatenate(yl), np.concatenate(rl), np.concatenate(cl)
    ok = np.array([not pd.isna(DOBRA.get(int(r))) for r in rl]) & ~np.isin(rl, list(TESTE))
    comps = np.array([alvos[j] for j in cl])
    return Xl[ok], yl[ok], cl[ok], np.array([int(DOBRA[int(r)]) for r in rl[ok]]), comps[ok]


def peso(y, comps):
    """Peso tres sobre o limite, caindo para um a cerca de trinta graus dele (Secao 23 da modelagem)."""
    lim = np.array([LIM.get(c, np.nan) for c in comps], float)
    return np.where(np.isnan(lim), 1.0, 1.0 + 2.0 * np.exp(-(np.nan_to_num(np.abs(y - lim), nan=1e9) / 15.0) ** 2))


def matriz(X, y, c, tr, alvo):
    """Entradas mais patamar e escala do componente, calculados apenas nas linhas tr."""
    p_ = pd.Series(y[tr]).groupby(c[tr]).mean()
    s_ = pd.Series(y[tr]).groupby(c[tr]).std().fillna(1.0).replace(0, 1.0)
    ger = float(y[tr].mean())
    return np.hstack([X[alvo], pd.Series(c[alvo]).map(p_).fillna(ger).to_numpy().reshape(-1, 1),
                      pd.Series(c[alvo]).map(s_).fillna(1.0).to_numpy().reshape(-1, 1)])


def metricas(y, p, co):
    ok = ~np.isnan(p); y, p, co = y[ok], p[ok], co[ok]
    d = p - y
    lim = np.array([LIM.get(x, np.nan) for x in co], float); m = ~np.isnan(lim)
    est = y[m] >= lim[m]; ala = p[m] + MARGEM >= lim[m]
    return {"MAE": np.abs(d).mean(), "RMSE": np.sqrt((d ** 2).mean()),
            "R2": 1 - (d ** 2).sum() / ((y - y.mean()) ** 2).sum(), "vies": d.mean(),
            "dentro10_%": 100 * np.mean(np.abs(d) / y < 0.10),
            "aprov_indevidas": (est & ~ala).sum(), "reprov_desnecessarias": (~est & ala).sum(),
            "deteccao_%": 100 * (est & ala).sum() / est.sum() if est.sum() else np.nan}


pd.set_option("display.width", 220)
print({a: len(entradas(a, "detalhado", "com")[1]) for a in AREAS})

## 1. Grade ampla por área, com escolha pelo aninhamento estrito

A formulação entregue escolhe, por área, entre poucos candidatos. Esta seção mede o que aconteceria se a escolha fosse bem mais ampla: vinte candidatos por área, formados por cinco estimadores (floresta, média entre floresta e gradiente, Extra Trees, média entre Extra Trees e gradiente, e gradiente, todos com trezentas árvores e valores padrão), duas codificações das perdas e o motivo do ensaio presente ou ausente.

A escolha é medida pelo aninhamento estrito. Em cada dobra externa, os vinte candidatos são comparados só com as dezesseis configurações de treino, numa validação interna de quatro blocos que recalcula o patamar e a escala em cada bloco, e o vencedor interno prevê a dobra externa. O número reportado já inclui, assim, o custo de escolher entre vinte opções. A mesma escolha é refeita restrita aos quatro candidatos da grade anterior à troca em Main Vent Sys, para comparar as duas grades nas mesmas previsões. Três sementes.

In [ ]:
ESTIMADORES = ["rf", "rf+gb", "et", "et+gb", "gb"]
VARIANTES = [(t, m) for t in ("detalhado", "agrupado") for m in ("com", "sem")]
CAND = [(e, t, m) for e in ESTIMADORES for (t, m) in VARIANTES]
CAND_ANTERIOR = [c for c in CAND if c[0] in ("rf", "rf+gb") and c[2] == "com"]


def ajusta_variante(X, y, c, co, tr, va, s):
    A, B = matriz(X, y, c, tr, tr), matriz(X, y, c, tr, va)
    w = peso(y[tr], co[tr])
    rf = RandomForestRegressor(n_estimators=300, random_state=s, n_jobs=1).fit(A, y[tr], sample_weight=w).predict(B)
    et = ExtraTreesRegressor(n_estimators=300, random_state=s, n_jobs=1).fit(A, y[tr], sample_weight=w).predict(B)
    gb = GradientBoostingRegressor(n_estimators=300, random_state=s).fit(A, y[tr], sample_weight=w).predict(B)
    return {"rf": rf, "et": et, "gb": gb, "rf+gb": (rf + gb) / 2, "et+gb": (et + gb) / 2}


def tarefa(area, s, k):
    D = {v: entradas(area, *v) for v in VARIANTES}
    y, c, g, co = D[VARIANTES[0]][1:]
    tr, va = g != k, g == k
    oof, err = {}, {cc: 0.0 for cc in CAND}
    for v in VARIANTES:
        X = D[v][0]
        p = ajusta_variante(X, y, c, co, tr, va, s)
        for e in ESTIMADORES:
            oof[(e,) + v] = p[e]
        Xs, ys, cs, gs, cos = X[tr], y[tr], c[tr], g[tr], co[tr]
        cfg = sorted(set(gs))
        for b in [cfg[i::N_BLOCOS] for i in range(N_BLOCOS)]:
            iv = np.isin(gs, b); it = ~iv
            if it.sum() < 30 or not iv.any():
                continue
            pi = ajusta_variante(Xs, ys, cs, cos, it, iv, s)
            for e in ESTIMADORES:
                err[(e,) + v] += np.abs(pi[e] - ys[iv]).sum()
    return area, s, k, np.where(va)[0], oof, err


t0 = time.time()
Y = {a: entradas(a, "detalhado", "com")[1:] for a in AREAS}   # y, c, g, comps
tarefas = [(a, s, k) for a in AREAS for s in SEMENTES for k in sorted(set(Y[a][2])) if (Y[a][2] != k).sum() >= 30]
RES = Parallel(n_jobs=-1)(delayed(tarefa)(*t) for t in tarefas)
OOF = {(a, s): {cc: np.full(len(Y[a][0]), np.nan) for cc in CAND} for a in AREAS for s in SEMENTES}
ERR = {}
for a, s, k, idx, oof, err in RES:
    for cc in CAND:
        OOF[(a, s)][cc][idx] = oof[cc]
    ERR[(a, s, k)] = err
print(f"{len(tarefas)} tarefas em {(time.time() - t0) / 60:.1f} min")

In [ ]:
EXTRA = {"main_vent_sys": [("et+gb", "detalhado", "com"), ("et+gb", "agrupado", "com")]}
ENTREGA_ANTERIOR = dict(ENTREGA, main_vent_sys=("rf+gb", "detalhado", "com"))


def formula(a, s, modo):
    y, c, g, co = Y[a]
    oof = OOF[(a, s)]
    out, esc = np.full(len(y), np.nan), []
    for k in sorted(set(g)):
        if (a, s, k) not in ERR:
            continue
        va = g == k
        if modo == "regra fixa, entrega anterior":
            cc = ENTREGA_ANTERIOR[a]
        elif modo == "regra fixa, entrega atual":
            cc = ENTREGA[a]
        elif modo == "estrito, grade anterior (4)":
            cc = min(CAND_ANTERIOR, key=lambda x: ERR[(a, s, k)][x])
        elif modo == "estrito, grade atual (4 ou 6)":
            cc = min(CAND_ANTERIOR + EXTRA.get(a, []), key=lambda x: ERR[(a, s, k)][x])
        elif modo == "estrito, grade ampla (20)":
            cc = min(CAND, key=lambda x: ERR[(a, s, k)][x])
        out[va] = oof[cc][va]; esc.append(cc)
    return out, esc


MODOS = ["regra fixa, entrega anterior", "regra fixa, entrega atual", "estrito, grade anterior (4)",
         "estrito, grade atual (4 ou 6)", "estrito, grade ampla (20)"]
AG, POR_AREA, ESC, ERRS = {}, {}, {m: {a: Counter() for a in AREAS} for m in MODOS}, {}
for m in MODOS:
    lin, ea = [], []
    for s in SEMENTES:
        ps = []
        for a in AREAS:
            p, e = formula(a, s, m)
            ps.append(p); ESC[m][a].update(e)
            POR_AREA.setdefault((m, a), []).append(metricas(Y[a][0], p, Y[a][3]))
        y_all = np.concatenate([Y[a][0] for a in AREAS]); co_all = np.concatenate([Y[a][3] for a in AREAS])
        lin.append(metricas(y_all, np.concatenate(ps), co_all)); ea.append(np.abs(np.concatenate(ps) - y_all))
    AG[m] = pd.DataFrame(lin).mean(); ERRS[m] = np.mean(ea, axis=0)

print("Agregado das sete areas, media de tres sementes:\n")
print(pd.DataFrame(AG).T.round(3).to_string())
ref = ERRS["estrito, grade anterior (4)"]
print("\nDiferenca de erro por leitura contra a grade anterior estrita (IC95 por bootstrap das leituras):")
for m in MODOS:
    d = ref - ERRS[m]
    bs = [np.random.default_rng(i).choice(d, len(d), replace=True).mean() for i in range(2000)]
    print(f"  {m:32s} {d.mean():+.3f} | [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]")
print("\nErro medio absoluto por area:")
print(pd.DataFrame({m: {a: pd.DataFrame(POR_AREA[(m, a)]).mean()["MAE"] for a in AREAS} for m in MODOS}).round(3).to_string())
print("\nAprovacoes indevidas por area:")
print(pd.DataFrame({m: {a: pd.DataFrame(POR_AREA[(m, a)]).mean()["aprov_indevidas"] for a in AREAS} for m in MODOS}).round(1).to_string())
print("\nCandidato mais escolhido pela grade ampla estrita, em 51 escolhas por area:")
for a in AREAS:
    print(f"  {a:14s} " + " ; ".join(f"{e}|{t}|motivo {mo}: {n}" for (e, t, mo), n in ESC["estrito, grade ampla (20)"][a].most_common(3)))

In [ ]:
# Cada candidato aplicado igual em todas as dobras, sem escolha, para leitura do porque.
tab = {a: {f"{e} | {t} | motivo {mo}": np.mean([np.nanmean(np.abs(OOF[(a, s)][(e, t, mo)] - Y[a][0])) for s in SEMENTES])
           for (e, t, mo) in CAND} for a in AREAS}
print("Erro medio absoluto de cada candidato, sem escolha, media de tres sementes:\n")
print(pd.DataFrame(tab).round(3).to_string())

# A mescla ET + GB em Coil Temps: menor erro medio, triagem pior.
for cc in [("rf+gb", "detalhado", "com"), ("et+gb", "detalhado", "com")]:
    m = pd.DataFrame([metricas(Y["coil_temps"][0], OOF[("coil_temps", s)][cc], Y["coil_temps"][3]) for s in SEMENTES]).mean()
    print(f"\nCoil Temps, {cc[0]} com perdas {cc[1]}: MAE {m['MAE']:.3f} | aprovacoes indevidas {m['aprov_indevidas']:.1f} "
          f"| deteccao {m['deteccao_%']:.1f}%")

### Leitura

A grade ampla piora o agregado quando a escolha é medida pelo aninhamento estrito. Com dezesseis configurações de treino, a validação interna não separa vinte candidatos, e a escolha oscila de dobra para dobra: em várias áreas o candidato mais escolhido vence em menos da metade das escolhas. O custo de escolher supera o ganho de ter mais opções.

A tabela de candidatos sem escolha mostra também o efeito do motivo do ensaio. Retirá-lo empata em Main Vent Sys e CT Knobs e piora muito as demais áreas, em IPC Comp em mais de dois graus.

Duas leituras por área orientaram a entrega. Em Main Vent Sys, a média entre Extra Trees e gradiente é melhor que a média entre floresta e gradiente em todas as variantes, e foi adotada como candidata daquela área na Seção 23 do caderno de modelagem; a área não tem leituras acima do limite, de modo que a triagem não muda. Em Coil Temps, a mesma mescla reduz o erro médio, mas aumenta as aprovações indevidas e reduz a detecção, e por isso não foi adotada. A troca em Main Vent Sys foi identificada olhando estes resultados, e pelo aninhamento estrito ela não altera o agregado de forma mensurável.

## 2. Bootstrap sobre a formulação entregue

Duas aplicações, ambas reamostrando configurações inteiras, e não leituras, pelo mesmo motivo da validação.

A primeira é o intervalo das métricas: as dezessete configurações são reamostradas com reposição cinco mil vezes sobre as previsões fora da dobra da regra fixa da entrega, calculadas na seção 1, sem novo treino. O intervalo mostra quanto o número mudaria com outra amostra de configurações.

A segunda é a validação por bootstrap: em cada uma de cinquenta repetições, o modelo é treinado numa amostra com reposição das configurações e avaliado nas que ficaram de fora. Como uma amostra com reposição contém, em média, cerca de onze configurações distintas, o erro é maior que o da validação por configuração, e a comparação entre os dois funciona como uma curva de aprendizado.

In [ ]:
BASE_B = {}
for s in SEMENTES:
    ys, ps, gs, cs, ar = [], [], [], [], []
    for a in AREAS:
        y, c, g, co = Y[a]
        ys.append(y); ps.append(formula(a, s, "regra fixa, entrega atual")[0]); gs.append(g); cs.append(co); ar.append(np.full(len(y), a))
    y, p, g, co, ar = map(np.concatenate, (ys, ps, gs, cs, ar))
    BASE_B[s] = dict(y=y, p=p, g=g, ar=ar, lim=np.array([LIM.get(x, np.nan) for x in co], float))
CONFIGS = sorted(set(BASE_B[42]["g"]))
IDX = {k: np.where(BASE_B[42]["g"] == k)[0] for k in CONFIGS}


def metricas_b(idx):
    out = []
    for s in SEMENTES:
        b = BASE_B[s]
        y, p, lim, ar = b["y"][idx], b["p"][idx], b["lim"][idx], b["ar"][idx]
        ok = ~np.isnan(lim) & ~np.isnan(p)
        est = y[ok] >= lim[ok]; ala = p[ok] + MARGEM >= lim[ok]
        m = {"MAE": np.nanmean(np.abs(p - y)), "aprov_indevidas": (est & ~ala).sum(),
             "deteccao_%": 100 * (est & ala).sum() / est.sum() if est.sum() else np.nan}
        for a in AREAS:
            k = ar == a
            m["MAE " + a] = np.nanmean(np.abs(p[k] - y[k])) if k.any() else np.nan
        out.append(m)
    return pd.DataFrame(out).mean()


ponto = metricas_b(np.concatenate([IDX[k] for k in CONFIGS]))
rng = np.random.default_rng(2026)
BOOT = pd.DataFrame([metricas_b(np.concatenate([IDX[k] for k in rng.choice(CONFIGS, len(CONFIGS), replace=True)]))
                     for _ in range(5000)])
print("Intervalo por bootstrap de configuracoes, regra fixa da entrega (5.000 reamostras):\n")
print(f"{'metrica':26s}{'valor':>9s}{'IC95 inf':>10s}{'IC95 sup':>10s}")
for col in ["MAE", "aprov_indevidas", "deteccao_%"] + ["MAE " + a for a in AREAS]:
    v = BOOT[col].dropna()
    print(f"{col:26s}{ponto[col]:9.3f}{np.percentile(v, 2.5):10.3f}{np.percentile(v, 97.5):10.3f}")

In [ ]:
DADOS_E = {a: entradas(a, ENTREGA[a][1], ENTREGA[a][2]) for a in AREAS}


def repeticao(b):
    r = np.random.default_rng(1000 + b)
    sorteio = r.choice(CONFIGS, len(CONFIGS), replace=True)
    fora = [k for k in CONFIGS if k not in set(sorteio)]
    linhas = []
    for a in AREAS:
        X, y, c, g, co = DADOS_E[a]
        tr_idx = np.concatenate([np.where(g == k)[0] for k in sorteio])
        va = np.isin(g, fora)
        if not va.any():
            continue
        Xt, yt, ct = X[tr_idx], y[tr_idx], c[tr_idx]
        todos = np.ones(len(yt), bool)
        A = matriz(Xt, yt, ct, todos, todos)
        p_ = pd.Series(yt).groupby(ct).mean(); s_ = pd.Series(yt).groupby(ct).std().fillna(1.0).replace(0, 1.0)
        B = np.hstack([X[va], pd.Series(c[va]).map(p_).fillna(float(yt.mean())).to_numpy().reshape(-1, 1),
                       pd.Series(c[va]).map(s_).fillna(1.0).to_numpy().reshape(-1, 1)])
        w = peso(yt, co[tr_idx])
        est = ENTREGA[a][0]
        base = ExtraTreesRegressor if est.startswith("et") else RandomForestRegressor
        p = base(n_estimators=300, random_state=b, n_jobs=1).fit(A, yt, sample_weight=w).predict(B)
        if est.endswith("+gb"):
            p = (p + GradientBoostingRegressor(n_estimators=300, random_state=b).fit(A, yt, sample_weight=w).predict(B)) / 2
        linhas.append(pd.DataFrame({"area": a, "y": y[va], "p": p, "lim": [LIM.get(x, np.nan) for x in co[va]]}))
    d = pd.concat(linhas); ok = d["lim"].notna()
    est_ = d.loc[ok, "y"] >= d.loc[ok, "lim"]; ala = d.loc[ok, "p"] + MARGEM >= d.loc[ok, "lim"]
    out = {"configs_fora": len(fora), "MAE": (d["p"] - d["y"]).abs().mean(),
           "deteccao_%": 100 * (est_ & ala).sum() / est_.sum() if est_.sum() else np.nan}
    for a in AREAS:
        k = d["area"] == a
        out["MAE " + a] = (d.loc[k, "p"] - d.loc[k, "y"]).abs().mean() if k.any() else np.nan
    return out


OOB = pd.DataFrame(Parallel(n_jobs=-1)(delayed(repeticao)(b) for b in range(50)))
print(f"Validacao por bootstrap, 50 repeticoes; configuracoes fora da bolsa: media {OOB['configs_fora'].mean():.1f}\n")
print(f"{'metrica':26s}{'media':>9s}{'P2,5':>9s}{'P97,5':>9s}")
for col in ["MAE", "deteccao_%"] + ["MAE " + a for a in AREAS]:
    v = OOB[col].dropna()
    print(f"{col:26s}{v.mean():9.3f}{np.percentile(v, 2.5):9.3f}{np.percentile(v, 97.5):9.3f}")

### Leitura

O intervalo das métricas é largo. Com outras dezessete configurações de ensaio, o mesmo modelo poderia medir de cerca de 4,3 a 7,9 graus no agregado, contra os 5,87 medidos, o que é o efeito numérico de a base ter só dezessete configurações distintas. Isso não invalida a comparação entre modelos, que é pareada e tem incerteza muito menor, mas indica que o número absoluto deve ser apresentado com essa margem.

A validação por bootstrap mede um erro maior que o da validação por configuração, porque cada modelo treina com cerca de onze configurações distintas. Lida junto com a validação por configuração, que treina com dezesseis, e com a de cinco dobras da Seção 22 do caderno de modelagem, que treina com cerca de catorze, ela forma uma curva de aprendizado: o erro ainda cai rápido a cada configuração nova, o que indica que o limite atual vem da quantidade de dados.

O bootstrap como técnica de aumento da base não se aplica: repetir ensaios não acrescenta informação, e a floresta já reamostra os dados internamente.

## 3. AutoML com FLAML

Em cada uma das dezessete dobras externas, o FLAML escolhe algoritmo e hiperparâmetros dentro do treino externo, entre LightGBM, XGBoost, floresta e Extra Trees, com validação interna agrupada por configuração em quatro blocos, métrica de erro absoluto médio e o peso da entrega. O modelo escolhido prevê a dobra retida. A referência é a regra fixa da entrega, na mesma semente.

Uma ressalva de desenho: dentro da validação interna do FLAML, o patamar e a escala vêm do treino externo inteiro, que inclui o bloco interno de validação. Isso afeta só a escolha, e não a avaliação externa.

In [ ]:
try:
    from flaml import AutoML
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "flaml[automl]"])
    from flaml import AutoML

ORCAMENTO_FLAML = 60     # segundos por dobra


def dobra_flaml(area, k):
    import warnings as _w
    _w.filterwarnings("ignore")
    from flaml import AutoML
    X, y, c, g, co = DADOS_E[area]
    tr, va = g != k, g == k
    A, B = matriz(X, y, c, tr, tr), matriz(X, y, c, tr, va)
    cfg = sorted(set(g[tr])); bloco = {cf: i % N_BLOCOS for i, cf in enumerate(cfg)}
    automl = AutoML()
    automl.fit(X_train=A, y_train=y[tr], task="regression", metric="mae", time_budget=ORCAMENTO_FLAML,
               estimator_list=["lgbm", "rf", "extra_tree", "xgboost", "xgb_limitdepth"], eval_method="cv",
               split_type="group", groups=np.array([bloco[x] for x in g[tr]]), n_splits=N_BLOCOS,
               sample_weight=peso(y[tr], co[tr]), seed=42, n_jobs=1, verbose=0)
    return area, np.where(va)[0], automl.predict(B), automl.best_estimator


t0 = time.time()
tar = [(a, k) for a in AREAS for k in sorted(set(DADOS_E[a][3])) if (DADOS_E[a][3] != k).sum() >= 30]
RF_ = Parallel(n_jobs=-1)(delayed(dobra_flaml)(a, k) for a, k in tar)
PF = {a: np.full(len(DADOS_E[a][1]), np.nan) for a in AREAS}
ESC_F = {a: Counter() for a in AREAS}
for a, idx, p, e in RF_:
    PF[a][idx] = p; ESC_F[a][e] += 1
lin = []
for a in AREAS:
    y, co = Y[a][0], Y[a][3]
    lin.append({"area": a, "modelo": "entrega", **metricas(y, formula(a, 42, "regra fixa, entrega atual")[0], co)})
    lin.append({"area": a, "modelo": "FLAML", **metricas(y, PF[a], co)})
y_all = np.concatenate([Y[a][0] for a in AREAS]); co_all = np.concatenate([Y[a][3] for a in AREAS])
lin.append({"area": "AGREGADO", "modelo": "entrega",
            **metricas(y_all, np.concatenate([formula(a, 42, "regra fixa, entrega atual")[0] for a in AREAS]), co_all)})
lin.append({"area": "AGREGADO", "modelo": "FLAML", **metricas(y_all, np.concatenate([PF[a] for a in AREAS]), co_all)})
print(f"{len(tar)} ajustes do FLAML em {(time.time() - t0) / 60:.1f} min\n")
print(pd.DataFrame(lin).round(3).to_string(index=False))
print("\nAlgoritmo escolhido pelo FLAML nas dobras:")
for a in AREAS:
    print(f"  {a:14s} {dict(ESC_F[a].most_common())}")

## 4. AutoML com PyCaret

O PyCaret compara cerca de vinte regressores próprios em cada dobra externa, com validação interna agrupada por configuração, e o melhor, ajustado no treino externo, prevê a dobra retida. Não se usa a finalização do PyCaret, que reajustaria o modelo incluindo a dobra retida. O PyCaret não aceita o peso próximo ao limite, de modo que a entrega é reportada com e sem o peso.

O PyCaret exige Python 3.11 ou anterior e fixa uma versão antiga do scikit-learn. A célula executa a análise se o pacote estiver instalado no ambiente; caso contrário, imprime o resultado registrado na execução feita em ambiente separado, com Python 3.11.16, PyCaret 3.3.2 e scikit-learn 1.4.2, em 24 de setembro de 2026.

In [ ]:
try:
    from pycaret.regression import RegressionExperiment
    TEM_PYCARET = True
except ImportError:
    TEM_PYCARET = False


def dobra_pycaret(area, k):
    from pycaret.regression import RegressionExperiment
    X, y, c, g, co = DADOS_E[area]
    tr, va = g != k, g == k
    A, B = matriz(X, y, c, tr, tr), matriz(X, y, c, tr, va)
    cols = [f"x{i}" for i in range(A.shape[1])]
    dtr = pd.DataFrame(A, columns=cols); dtr["y"] = y[tr]
    dva = pd.DataFrame(B, columns=cols, index=range(len(A), len(A) + len(B))); dva["y"] = y[va]
    cfg = sorted(set(g[tr])); bloco = {cf: i % N_BLOCOS for i, cf in enumerate(cfg)}
    exp = RegressionExperiment()
    exp.setup(data=dtr, test_data=dva, target="y", preprocess=False, fold_strategy="groupkfold", fold=N_BLOCOS,
              fold_groups=pd.Series([bloco[x] for x in g[tr]], index=dtr.index), session_id=42, n_jobs=1,
              verbose=False, html=False, log_experiment=False)
    melhor = exp.compare_models(sort="MAE", n_select=1, verbose=False, errors="ignore")
    return area, np.where(va)[0], melhor.predict(dva[cols]), type(melhor).__name__


if TEM_PYCARET:
    RP = Parallel(n_jobs=-1)(delayed(dobra_pycaret)(a, k) for a, k in tar)
    PP = {a: np.full(len(DADOS_E[a][1]), np.nan) for a in AREAS}
    for a, idx, p, e in RP:
        PP[a][idx] = p
    lin = [{"area": a, "MAE PyCaret": metricas(Y[a][0], PP[a], Y[a][3])["MAE"]} for a in AREAS]
    print(pd.DataFrame(lin).round(3).to_string(index=False))
else:
    print("PyCaret nao instalado neste ambiente. Resultado registrado em ambiente Python 3.11, 24/09/2026,")
    print("com a regra da entrega anterior a troca em Main Vent Sys (erro medio absoluto, graus Celsius):\n")
    print("  area            entrega com peso   entrega sem peso   PyCaret")
    for a, e1, e2, pc in [("oven_hmi", 2.353, 2.347, 2.541), ("ct_knobs", 2.990, 2.979, 2.956),
                          ("main_vent_sys", 3.898, 3.881, 4.691), ("ipc_cooling", 3.775, 3.755, 3.956),
                          ("mainback", 4.568, 4.499, 4.802), ("ipc_comp", 6.573, 6.643, 6.976),
                          ("coil_temps", 10.192, 10.339, 11.205), ("AGREGADO", 5.853, 5.891, 6.267)]:
        print(f"  {a:14s} {e1:16.3f} {e2:18.3f} {pc:9.3f}")
    print("\n  aprovacoes indevidas no agregado: entrega com peso 27, sem peso 34, PyCaret 26")

### Leitura das duas ferramentas de AutoML

As duas ferramentas ficam atrás da entrega no agregado. O FLAML mede 6,37 graus contra 5,85 da entrega, com 33 aprovações indevidas contra 25, e é pior em seis das sete áreas. A exceção é Mainback, onde erra 4,42 contra 4,59 graus e deixa passar quatro estouros contra seis, resultado de uma única semente e sem repetição, que fica registrado como candidato a teste, e não como mudança. O PyCaret empata em CT Knobs, onde escolhe a própria floresta em todas as dobras, e é pior nas demais. Nas duas, o algoritmo escolhido muda de dobra para dobra, sinal de candidatos empatados: com quarenta e seis ensaios em dezessete configurações, a busca automática ajusta ruído, e a floresta com valores padrão se sai melhor. Os skills de AutoML da DataRobot não foram usados, porque enviam os dados a um serviço externo, o que conflita com o compromisso de proteção de dados com a parceira.

## 5. Oven HMI: o efeito de contar duas vezes o mesmo sensor

A base da terceira sprint tinha, no Oven HMI, o sensor Thermistor, que é cópia exata do Microchip, com correlação 1 e diferença máxima zero nas 45 leituras. A base atual o descarta. Configurações de hiperparâmetros escolhidas naquela base, como o gradiente e o Extra Trees da busca aleatória da terceira sprint, apareciam com erro menor que a entrega, e esta seção mostra de onde vinha a diferença.

A contagem dupla é reproduzida a partir da própria base desta pasta, acrescentando uma cópia do Microchip como quarto componente. Cada modelo é avaliado de duas formas: nas 180 leituras, como seria na base antiga, e nas 135 leituras distintas.

In [ ]:
JULIA_GB = dict(learning_rate=0.010, n_estimators=235, max_depth=6, min_samples_leaf=9, subsample=0.976)
JULIA_ET = dict(n_estimators=300, max_depth=29, max_features=0.72, min_samples_leaf=9)
X0, y0, c0, g0, co0 = entradas("oven_hmi", "agrupado", "com")
mc = co0 == "A33/Bellatrix Microchip"
# quarto componente: copia do Microchip, com indice de componente proprio
X2 = np.vstack([X0, X0[mc]]); y2 = np.concatenate([y0, y0[mc]])
c2 = np.concatenate([c0, np.full(mc.sum(), c0.max() + 1)]); g2 = np.concatenate([g0, g0[mc]])
co2 = np.concatenate([co0, np.full(mc.sum(), "A33/Bellatrix Microchip")])
distintas = np.concatenate([np.ones(len(y0), bool), np.zeros(mc.sum(), bool)])

MOD_O = {"entrega: floresta": lambda s: RandomForestRegressor(n_estimators=300, random_state=s, n_jobs=-1),
         "gradiente da busca da terceira sprint": lambda s: GradientBoostingRegressor(random_state=s, **JULIA_GB),
         "Extra Trees da busca da terceira sprint": lambda s: ExtraTreesRegressor(random_state=s, n_jobs=-1, **JULIA_ET)}
lin = []
for nome, fab in MOD_O.items():
    for base, (X, y, c, g, co) in (("135 leituras", (X0, y0, c0, g0, co0)), ("180 leituras", (X2, y2, c2, g2, co2))):
        ms180, ms135 = [], []
        for s in SEMENTES:
            p = np.full(len(y), np.nan)
            for k in sorted(set(g)):
                tr, va = g != k, g == k
                p[va] = fab(s).fit(matriz(X, y, c, tr, tr), y[tr], sample_weight=peso(y[tr], co[tr])).predict(matriz(X, y, c, tr, va))
            if base == "180 leituras":
                ms180.append(metricas(y, p, co)); ms135.append(metricas(y[distintas], p[distintas], co[distintas]))
            else:
                ms135.append(metricas(y, p, co))
        linha = {"modelo": nome, "treinado em": base, "MAE nas 135 distintas": pd.DataFrame(ms135).mean()["MAE"]}
        if ms180:
            linha["MAE nas 180"] = pd.DataFrame(ms180).mean()["MAE"]
        lin.append(linha)
print(pd.DataFrame(lin).round(3).to_string(index=False))

### Leitura

Nas mesmas 135 leituras distintas, a entrega tem o menor erro, 2,32 graus, em todas as combinações. Treinar com a cópia do Microchip piora a floresta e melhora um pouco o gradiente e o Extra Trees, mas nenhum deles alcança a entrega, nem avaliado nas 180 leituras, em que medem entre 2,41 e 2,42.

A contagem dupla, reproduzida sozinha sobre a base atual, não explica, portanto, toda a vantagem que esses modelos exibiam na base da terceira sprint, onde mediam entre 2,28 e 2,32 graus. O restante vem das outras diferenças daquela base, que codificava o motivo do ensaio pelos indicadores anteriores e as perdas pelo perfil, e do fato de as configurações terem sido escolhidas e avaliadas nas mesmas dobras: refeita dentro das dobras de treino, a busca que as produziu mede entre 2,6 e 2,7 graus. Por essas razões, o Oven HMI da entrega foi mantido.